In [21]:
from pathlib import Path

import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd()

# If the notebook starts inside the notebooks folder, move one level up.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DB_PATH = PROJECT_ROOT / "db" / "jobs.duckdb"

print("Project root:", PROJECT_ROOT)
print("Database path:", DB_PATH)
print("Database exists:", DB_PATH.exists())

con = duckdb.connect(str(DB_PATH))
con.execute("SHOW TABLES").df()

summary_query = """
SELECT
    COUNT(DISTINCT job_post_id) AS total_unique_job_postings,
    COUNT(DISTINCT category_name) AS total_categories,
    COUNT(DISTINCT company_name) AS total_companies,
    MIN(new_posting_date) AS earliest_posting_date,
    MAX(new_posting_date) AS latest_posting_date
FROM vw_career_coach_jobs;
"""

database_summary = con.execute(summary_query).df()
display(database_summary)



# Limit all queries to 10 for now first

# query = """
# SELECT *
# FROM vw_career_coach_jobs
# LIMIT 10;
# """
# preview_df = con.execute(query).df()
# preview_df

query = """
SELECT
    category_name,
    COUNT(DISTINCT job_post_id) AS total_job_postings
FROM vw_career_coach_jobs
WHERE category_name IS NOT NULL
GROUP BY category_name
ORDER BY total_job_postings DESC
LIMIT 10;
"""
top_categories_by_postings = con.execute(query).df()

query = """
SELECT
    category_name,
    SUM(number_of_vacancies) AS total_vacancies
FROM vw_career_coach_jobs
WHERE category_name IS NOT NULL
GROUP BY category_name
ORDER BY total_vacancies DESC
LIMIT 10;
"""
top_categories_by_vacancies = con.execute(query).df()

query = """
SELECT
    category_name,
    MEDIAN(salary_minimum) AS median_salary_minimum,
    MEDIAN(salary_maximum) AS median_salary_maximum,
    MEDIAN(average_salary) AS median_average_salary
FROM vw_career_coach_jobs
WHERE category_name IS NOT NULL
  AND average_salary IS NOT NULL
GROUP BY category_name
ORDER BY median_average_salary DESC
LIMIT 10;
"""
salary_by_category = con.execute(query).df()

salary_by_category[
    [
        "category_name",
        "median_salary_minimum",
        "median_salary_maximum",
        "median_average_salary"
    ]
]

query = """
SELECT
    category_name,
    COUNT(DISTINCT job_post_id) AS total_job_postings,
    MEDIAN(minimum_years_experience) AS median_min_years_experience,
    AVG(minimum_years_experience) AS average_min_years_experience
FROM vw_career_coach_jobs
WHERE category_name IS NOT NULL
  AND minimum_years_experience IS NOT NULL
GROUP BY category_name
ORDER BY median_min_years_experience ASC, total_job_postings DESC
LIMIT 10;
"""
experience_by_category = con.execute(query).df()


query = """
SELECT
    category_name,
    COUNT(DISTINCT job_post_id) AS entry_level_job_postings,
    MEDIAN(average_salary) AS median_average_salary,
    MEDIAN(minimum_years_experience) AS median_min_years_experience
FROM vw_career_coach_jobs
WHERE category_name IS NOT NULL
  AND minimum_years_experience <= 1
GROUP BY category_name
ORDER BY entry_level_job_postings DESC
LIMIT 10;
"""

entry_level_friendly_categories = con.execute(query).df()


display(top_categories_by_vacancies)
display(top_categories_by_postings)
display(salary_by_category)
display(experience_by_category)
display(entry_level_friendly_categories)





Project root: /home/kennywong/code/ntu-sctp/repos/dsai-module1-individual-project
Database path: /home/kennywong/code/ntu-sctp/repos/dsai-module1-individual-project/db/jobs.duckdb
Database exists: True


,total_unique_job_postings,total_categories,total_companies,earliest_posting_date,latest_posting_date
0,1044597,43,53151,2023-02-24,2024-05-29


,category_name,total_vacancies
0,Customer Service,417980.0
1,Information Technology,316470.0
2,F&B,308650.0
3,Others,300759.0
4,Admin / Secretarial,295225.0
5,Sales / Retail,285694.0
6,Engineering,259883.0
7,Healthcare / Pharmaceutical,229960.0
8,Education and Training,222040.0
9,Human Resources,194629.0


,category_name,total_job_postings
0,Information Technology,140866
1,Engineering,136372
2,Admin / Secretarial,117854
3,Customer Service,111785
4,Others,106101
5,Sales / Retail,105067
6,Building and Construction,84034
7,Accounting / Auditing / Taxation,78648
8,F&B,73731
9,Logistics / Supply Chain,69193


,category_name,median_salary_minimum,median_salary_maximum,median_average_salary
0,Information Technology,5000.0,8000.0,6500.0
1,Risk Management,4500.0,7000.0,6000.0
2,Banking and Finance,4500.0,6500.0,5500.0
3,Insurance,3500.0,5500.0,4750.0
4,Consulting,3500.0,6000.0,4725.0
5,Sciences / Laboratory / R&D,3500.0,5100.0,4500.0
6,Engineering,3500.0,5000.0,4250.0
7,Professional Services,3300.0,5000.0,4150.0
8,Telecommunications,3000.0,5000.0,4150.0
9,Personal Care / Beauty,3000.0,5000.0,4000.0


,category_name,total_job_postings,median_min_years_experience,average_min_years_experience
0,Admin / Secretarial,117854,1.0,1.744421
1,Customer Service,111785,1.0,1.684815
2,Education and Training,44877,1.0,1.832453
3,General Work,30140,1.0,1.898109
4,Events / Promotions,16993,1.0,1.806803
5,Travel / Tourism,6549,1.0,2.187204
6,Security and Investigation,5461,1.0,2.035158
7,Others,106101,2.0,2.197180
8,Sales / Retail,105067,2.0,2.176668
9,Accounting / Auditing / Taxation,78648,2.0,2.798762


,category_name,entry_level_job_postings,median_average_salary,median_min_years_experience
0,Admin / Secretarial,67727,2600.0,1.0
1,Customer Service,64780,2750.0,1.0
2,Others,50685,2850.0,1.0
3,Sales / Retail,50420,3100.0,1.0
4,Engineering,35601,3400.0,1.0
5,Logistics / Supply Chain,29409,2600.0,1.0
6,F&B,28502,2550.0,1.0
7,Information Technology,27868,3750.0,1.0
8,Accounting / Auditing / Taxation,25298,3000.0,1.0
9,Human Resources,24660,3100.0,1.0
